In [1]:
import torch
from transformers import RobertaTokenizer,RobertaModel,RobertaForMaskedLM, pipeline, BertForMaskedLM, BertTokenizer, BertModel, AlbertConfig, AlbertModel, AlbertTokenizer, AlbertForMaskedLM , TFAutoModel, AutoModelForMaskedLM, AutoTokenizer, AutoModelForCausalLM
import math
import numpy as np
import json
import random
import pandas as pd
import tensorflow as tf
from torch.nn import CrossEntropyLoss
from tqdm.notebook import tqdm
import re
from scipy.optimize import linear_sum_assignment

import SMI_Methods

In [2]:
import logging
import os
import sys
logging.basicConfig(level=logging.INFO)
import warnings
warnings.filterwarnings('ignore')

In [30]:
# Dictionary to map Czech characters to their Polish equivalents.
CZECH_TO_POLISH = {
    'č': 'cz', 
    'š': 'sz', 
    'ř': 'rz', 
    'ž': 'ż', 
    'ý': 'y', 
    'á': 'a', 
    'í': 'i', 
    'é': 'e', 
    'ě': 'e', 
    'ú': 'u', 
    'ů': 'u',
    'v': 'w'
}

POLISH_TO_CZECH = {
    'ą': 'o',
    'ć': 'č',
    'ę': 'e',
    'ł': 'l',
    'ń': 'n',
    'ó': 'u',
    'ś': 'š',
    'ż': 'ž',
    'ź': 'ž',
    'w': 'v',
    'cz': 'č',
    'sz': 'š',
    'rz': 'ř',
    'dz': 'dz',
    'dź': 'ď',
    'dż': 'dž',
    'Cz': 'č',
    'Sz': 'š',
    'Rz': 'ř',
    'Dz': 'dz',
    'Dź': 'ď',
    'Dż': 'dž',
}

def is_list(element):
    """Check if the given element is a list."""
    return isinstance(element, list)

def czech_to_polish(text):
    """Transcribes a Czech text into Polish phonetically."""
    for czech_char, polish_char in CZECH_TO_POLISH.items():
        text = text.replace(czech_char, polish_char)
        text = text.replace(czech_char.upper(), polish_char.upper())  # Replace uppercase characters.
    return text

def polish_to_czech(text):
    """Transcribes a Polish text into Czech phonetically."""
    # We need to replace longer substrings first to avoid partial matches.
    for polish_char, czech_char in sorted(POLISH_TO_CZECH.items(), key=lambda x: -len(x[0])):
        text = text.replace(polish_char, czech_char)
        text = text.replace(polish_char.upper(), czech_char.upper())  # Replace uppercase characters.
    return text

def replace_in_nested_list(nested_list, direction="ctop"):
    """
    Perform the replacement operation on the deepest sublists of arbitrary depth.
    :param nested_list: The nested list to operate on.
    :param direction: The direction of the transcription ("ctop" for Czech to Polish,
                      "ptoc" for Polish to Czech).
    :return: The nested list with replaced strings.
    """
    if not is_list(nested_list):
        return czech_to_polish(nested_list) if direction == "ctop" else polish_to_czech(nested_list)
    return [replace_in_nested_list(sublist, direction) for sublist in nested_list]


In [31]:
def count_tokens(model, tokenizer, data):
    r = 0
    for d in data:
        for s in d:
            for l in s:
                r += len(tokenizer.tokenize(l))
    return r

def see_tokens(model, tokenizer, data):
    r = 0
    for d in data:
        for s in d:
            for l in range(1):
                print(tokenizer.tokenize(s[l]))

In [32]:
def run_tests(model, tokenizer, lang, transliterate):
    word_list = SMI_Methods.prep_words(lang)
    data = SMI_Methods.prep_data(lang, word_list)
    
       
    word_list = replace_in_nested_list(word_list, direction = transliterate)
    data = replace_in_nested_list(data, direction = transliterate)
    lang = transliterate
    
    token_count = count_tokens(model, tokenizer, data)
    
    results = []
    for desc, d in zip(["1 Line Data", "3 Line Data", "3 Line Data (unfilled)"], [data[0], data[1], data[2]]):
        scores_line = [0,0,0,0]
        for i in range(4):
            sc = SMI_Methods.score_model(model, tokenizer, d[i], word_list[i])
            scores_line[0] += sc[0]
            scores_line[1] += sc[1]
            scores_line[2] += sc[2]
            scores_line[3] += sc[3]
        results.append({
            "Language": lang,
            "Description": desc,
            "Token Count": token_count,
            "Top 1 Score": scores_line[0]/4,
            "Top 3 Score": scores_line[1]/4,
            "COS": scores_line[2]/4,
            "GAS": scores_line[3]/4
        })

    df = pd.DataFrame(results)
    return df


In [33]:
tokenizer = AutoTokenizer.from_pretrained("sdadas/polish-gpt2-small")
model = AutoModelForCausalLM.from_pretrained("sdadas/polish-gpt2-small").cuda()
df = run_tests(model, tokenizer, "Czech", "ctop")
df.to_csv("Outputs/Transliterated_CtoP.csv")

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

In [29]:
tokenizer = AutoTokenizer.from_pretrained("spital/gpt2-small-czech-cs")
model = AutoModelForCausalLM.from_pretrained("spital/gpt2-small-czech-cs").cuda()
df = run_tests(model, tokenizer, "Polish","ptoc")
df.to_csv("Outputs/Transliterated_PtoC.csv")

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

In [20]:
tokenizer = AutoTokenizer.from_pretrained("spital/gpt2-small-czech-cs")
model = AutoModelForCausalLM.from_pretrained("spital/gpt2-small-czech-cs").cuda()
word_list = SMI_Methods.prep_words("Polish")
data = SMI_Methods.prep_data("Polish", word_list)
print(data[0][0])
print(count_tokens(model, tokenizer, data))
see_tokens(model, tokenizer, data)
print("-------------------------------------------------------------------------------")
data = replace_in_nested_list(data, "ptoc")
print(data[0][0])
print(count_tokens(model, tokenizer, data))
see_tokens(model, tokenizer, data)
print("-------------------------------------------------------------------------------")
print("-------------------------------------------------------------------------------")
print("-------------------------------------------------------------------------------")
word_list = SMI_Methods.prep_words("Czech")
data = SMI_Methods.prep_data("Czech", word_list)
print(data[0][0])
print(count_tokens(model, tokenizer, data))
see_tokens(model, tokenizer, data)

['Rodzice, których dzieci wykazują szczególne zainteresowanie jakimś rodzajem sportu, mają do podjęcia trudną decyzję. Czy powinni oni pozwolić swoim dzieciom na trenowanie, by {} się one najlepszymi sportowcami? \n', 'Dla wielu {} to oznacza rozpoczęcie w bardzo młodym wieku.\n', 'Praca w szkole, spotykanie się ze znajomymi I inne zainteresowania muszą zejść na {} plan.\n', 'Bardzo {} jest wyjaśnić małym dzieciom dlaczego muszą trenować pięć godzin dziennie.\n', 'To dotyczy także weekendów, kiedy większość ich przyjaciół się {}.\n', 'Inny problem to oczywiście pieniądze. W wielu {} rząd udostępnia pieniądze na treningi dla najlepszych młodych sportowców.\n', 'Jeśli ta pomoc jest niedostępna, rodzice muszą znaleźć czas i {}, aby wesprzeć swoje dzieci.\n', 'Odzież sportowa, dowozy na zawody, specjalistyczne wyposażenie itp. mogą być bardzo {}.\n', 'W zrozumiały sposób wielu rodziców niepokoi się, że to niebezpieczne, żeby zaczynać poważne treningi sportowe w tak wczesnym wieku. Niektórz

0